## Day 1 - API Data Analysis Using Python Structures

In [ ]:
!uv pip install matplotlib

In [ ]:
!uv pip install requests matplotlib pandas numpy


In [ ]:
from pathlib import Path

import requests
import pandas as pd
import matplotlib.pyplot as plt

API_URL = "https://jsonplaceholder.typicode.com/users"

In [ ]:
def get_customers(url: str = API_URL) -> list[dict]:
    """Fetch the customer list. Report failures clearly instead of crashing."""
    try:
        response = requests.get(url, timeout=10)
    except requests.exceptions.Timeout:
        print("The request timed out. Try again later.")
        return []
    except requests.exceptions.ConnectionError:
        print("Could not connect. Check your internet connection.")
        return []
    except requests.exceptions.RequestException as exc:
        print(f"The request failed: {exc}")
        return []

    if response.status_code != 200:
        print(f"The API returned status {response.status_code}, expected 200.")
        return []

    try:
        return response.json()
    except ValueError:
        print("The API did not return valid JSON.")
        return []

**What this cell does:** calls the function, stores the result in `customers` and prints how many customers came back.

In [ ]:
customers = get_customers()
print(len(customers))

Defines `explore`, which prints every key of one customer record together with its Python type, indented to show how deep it sits.

Each customer is a dict. Some values inside it are dicts too (`address`, `company`) and `address` even contains another dict (`geo`). Like folders inside folders.

When `explore` meets a dict, it calls *itself* on that inner dict with `indent + 1`. That is why each level is indented further than the one above.

**You should see:** `address: dict` followed by indented `street`, `suite`, `city`, `zipcode` and `geo: dict` (with `lat` and `lng` indented even further) then `company: dict` with `name`, `catchPhrase` and `bs`.


In [ ]:
def explore(record: dict, indent: int = 0) -> None:
    """Print every key in a (possibly nested) record with its type, indented by depth."""
    for key, value in record.items():
        print("  " * indent + f"{key}: {type(value).__name__}")
        if isinstance(value, dict):
            explore(value, indent + 1)

explore(customers[0])

In [ ]:
def extract_attributes(customers: list[dict]) -> list[dict]:
    """Return one flat dict per customer."""
    rows = []
    for c in customers:
        email = c["email"]
        rows.append({
            "customer_id": c["id"],
            "name": c["name"],
            "username": c["username"],
            "email": email,
            "email_domain": email.split(".")[-1],
            "phone": c["phone"],
            "city": c["address"]["city"],
            "zipcode": c["address"]["zipcode"],
            "latitude": float(c["address"]["geo"]["lat"]),
            "longitude": float(c["address"]["geo"]["lng"]),
            "company": c["company"]["name"],
            "website": c["website"],
        })
    return rows

rows = extract_attributes(customers)
rows[1]

**What this cell does:** defines `unique_cities`, then prints how many distinct cities there are next to the number of customers.

**Key idea: sets.** A set stores each value only once, so duplicates vanish automatically. If three customers live in Nairobi, the set holds `"Nairobi"` just once.

In [ ]:
def unique_cities(customers: list[dict]) -> set[str]:
    """Return the set of distinct cities."""
    return {c["address"]["city"] for c in customers}

cities = unique_cities(customers)
print(f"{len(cities)} unique cities for {len(customers)} customers")

## Count customers by city and by email domain


In [ ]:
def sort_counts(counts: dict[str, int]) -> dict[str, int]:
    """Most common first; ties in alphabetical order."""
    return dict(sorted(counts.items(), key=lambda item: (-item[1], item[0])))


def city_counts(customers: list[dict]) -> dict[str, int]:
    """Count customers per city using a loop."""
    counts = {}
    for c in customers:
        city = c["address"]["city"]
        counts[city] = counts.get(city, 0) + 1
    return sort_counts(counts)


def email_domain_counts(customers: list[dict]) -> dict[str, int]:
    """Count customers per email top-level domain using a dict comprehension."""
    domains = [c["email"].split(".")[-1] for c in customers]
    counts = {d: domains.count(d) for d in set(domains)}
    return sort_counts(counts)


by_city = city_counts(customers)
by_domain = email_domain_counts(customers)
print(by_city)
print(by_domain)

## Move the data into pandas and cross-check

turns the list of flat dicts into a pandas DataFrame (a table), prints its size and column types, and checks that pandas gives the same counts as our hand-written code.

In [ ]:
def to_dataframe(rows: list[dict]) -> pd.DataFrame:
    return pd.DataFrame(rows)

df = to_dataframe(rows)
print(f"df.shape: {df.shape}\n")
print(f"df.dtypes: \n{df.dtypes}\n")

print("pandas on cities :", df["city"].value_counts().to_dict() == by_city)
print("pandas on domains:", df.groupby("email_domain").size().to_dict() == by_domain)

## Exporting the results to CSV


In [ ]:
from pathlib import Path


Path("output").mkdir(exist_ok=True)

df.to_csv("output/customers.csv", index=False)

pd.DataFrame(list(by_city.items()), columns=["city", "customer_count"]) \
    .to_csv("output/city_counts.csv", index=False)

## Drawing the charts

**What this cell does:** defines `plot_distribution`, which draws a bar chart from a counts dict and saves it as a PNG. It then makes two charts: customers per city, and customers per email domain.

The city chart is ten bars of equal height, and the domain chart has one tall `biz` bar. Both PNG files are saved in `output` folder.

In [ ]:
def plot_distribution(counts, path, title="Customers per city", xlabel="City"):
    """Save a labelled bar chart to path."""
    plt.figure(figsize=(9, 5))
    plt.bar(list(counts.keys()), list(counts.values()), color="#2a7de1")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Number of customers")
    plt.yticks(range(0, max(counts.values()) + 1))   # whole customers only
    plt.xticks(rotation=45, ha="right")              # readable labels
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()
    plt.close()

plot_distribution(by_city, "output/customer_distribution.png")
plot_distribution(by_domain, "output/email_domain_distribution.png",
                  title="Customers per email domain",
                  xlabel="Email top-level domain")

## Findings summary

Here every city has 1 customer, so the "most common city" is just whichever comes first alphabetically.

In [ ]:
top_city, top_n = next(iter(by_city.items()))
print(f"Customers retrieved : {len(customers)}")
print(f"Unique cities       : {len(cities)}")
print(f"Most common city    : {top_city} ({top_n})")
print(f"Email domains       : {by_domain}")
print("Chart saved to      : output/customer_distribution.png")